# 10 - Secure Multi-Tenant AI Gateway
## IDOR / BOLA meets AI Tenant Isolation

Covers the **Secure AI Architecture** part of the syllabus that had no hands-on demo yet: auth, RBAC, secrets management, rate limiting, tenant isolation, audit logging — all in one realistic scenario instead of a generic login form.

## The scenario

One AI backend serves three departments (tenants): **legal, finance, hr**. Each has its own confidential context (like a per-tenant secret/system prompt pulled from a secrets manager). Each API key belongs to exactly one tenant and one role (`admin`, `analyst`, `viewer`).

## The vulnerability class: BOLA / IDOR

**Broken Object-Level Authorization** (OWASP API Security Top 10, API1:2023) happens when a system checks *authentication* (is this a valid key?) but forgets to check *authorization* for the specific object being requested (does this key actually own THIS tenant's data?).

In classic web apps this looks like `/invoices/1234` where changing the ID lets you see someone else's invoice. In this AI gateway, the "object" is a **tenant's confidential context** injected into the LLM's system prompt — same bug, new surface.

This is also **OWASP LLM02: Sensitive Information Disclosure** — the model itself behaves correctly, but the application handed it the wrong tenant's secrets to begin with.

## Vulnerable code (the bug)

```python
identity = API_KEYS.get(req.api_key)      # checks the key is VALID
if not identity:
    return {"error": "invalid api key"}

# BUG: uses the CLIENT-SUPPLIED tenant_id, not identity['tenant']
answer = ask_groq(req.tenant_id, req.question)
```

Any authenticated key can read any tenant's secrets just by changing one field in the request body.

## The fix

```python
identity = API_KEYS.get(req.api_key)
real_tenant = identity["tenant"]          # derived from the KEY, never from the client
cross_tenant_attempt = req.tenant_id != real_tenant   # log it, don't trust it
answer = ask_groq(real_tenant, req.question)
```

**The golden rule this demonstrates:** never trust a client-supplied identity or object reference for an access decision. Always derive it from something you authenticated yourself (the API key's own record, a session, a signed JWT claim) — never from a field the caller can just type in.

## The other 3 pieces (also in `main.py`)

- **Role-based rate limiting** - `admin` gets 20 req/min, `analyst` 10, `viewer` 3. Stops a low-privilege key from being used to hammer the backend or scrape data slowly.
- **Tenant-scoped secrets** - `TENANT_SECRETS` simulates a secrets manager: each tenant's confidential context is isolated, never merged into one global prompt.
- **Tenant-scoped audit logging** - even the audit log itself respects tenant isolation: an admin only ever sees their own tenant's log entries, not everyone's.

## Try it yourself

Run the app (`uvicorn main:app --reload --port 8004` + `streamlit run app.py`), log in as **Finance - Analyst**, and in the `tenant_id` dropdown pick **legal** instead of finance.

- In **Vulnerable** mode: you'll get Legal's confidential settlement data back.
- In **Secure** mode: you'll still get an answer, but it will be Finance's own data, and the UI will tell you the cross-tenant attempt was blocked.

## Interview questions for this module

- What is BOLA/IDOR, and how does it apply to an AI/LLM backend, not just a REST API?
- Why is checking "is this API key valid" not enough — what else must you check?
- How would you enforce tenant isolation in a multi-tenant RAG or agent system?
- Why should an audit log itself be access-controlled and scoped, not a flat shared log?
- How would you rate-limit different roles differently, and why does that matter for cost and abuse control in an LLM-backed app?